## Step 4 — create initial zones
**# of cells in notebook:** 1

**Purpose:** Create initial analysis zones. This script takes `roads_9`, water lines, admin boundaries, and the extent hull, merges them, and treats the resulting enclosed polygons as 'zones'. 

**Input:**

- a geodatabase with: `roads_9`, `extent_hull`
- a country level geopackage from Geofabrik containing a `gis_osm_waterways_free` layer
- a folder with a contry admin boundaries shapefile downloaded from www.geoboundaries.org
- a 'zones' geodatabase

**Output:** `zones_1`

**Main logic:**

1. clip waterways by `extent_hull`
2. select admin bondaries that intersect `extent_hull` and convert to line
3. convert `extent_hull` to line
4. merge all line features, clip to `extnet_hull`, and polygonize

In [ ]:
import arcpy
import os
import time
import traceback

# ============================================================
# USER INPUTS
# ============================================================

# Geodatabases
roads_gdb = r"E:\World Bank deliverbale 1\_analysis\roads\roads.gdb"
zones_gdb = r"E:\World Bank deliverbale 1\_analysis\zones\zones.gdb"

# Existing roads / extent inputs
roads_9_fc = os.path.join(roads_gdb, "roads_9")
extent_hull_fc = os.path.join(roads_gdb, "extent_hull")

# Raw OSM waterways input
raw_waterways_fc = (
    r"C:\Users\Alex Blei\Downloads\south-sudan-260512-free.gpkg"
    r"\south-sudan.gpkg"
    r"\main.gis_osm_waterways_free"
)

# Raw geoBoundaries polygon input
raw_geoboundaries_fc = (
    r"C:\Users\Alex Blei\Downloads\geoBoundaries-SSD-ADM2-all"
    r"\geoBoundaries-SSD-ADM2.shp"
)

# Outputs to create in zones.gdb
waterways_clip_fc = os.path.join(zones_gdb, "waterways_clip")
geoboundaries_selection_fc = os.path.join(zones_gdb, "geoboundaries_selection")
geoboundaries_line_fc = os.path.join(zones_gdb, "geoboundaries_line")
extent_hull_line_fc = os.path.join(zones_gdb, "extent_hull_line")
zones_1_fc = os.path.join(zones_gdb, "zones_1")


# ============================================================
# SETTINGS
# ============================================================

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

scratch_gdb = arcpy.env.scratchGDB

merged_lines_fc = os.path.join(scratch_gdb, "tmp_zones_merged_lines")
clipped_merged_lines_fc = os.path.join(scratch_gdb, "tmp_zones_clipped_merged_lines")

geoboundaries_lyr = "geoboundaries_lyr"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def msg(text):
    print(text)
    arcpy.AddMessage(text)


def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)


def check_exists(path, label):
    if not arcpy.Exists(path):
        raise FileNotFoundError(f"{label} does not exist:\n{path}")


def describe_basic(path, label):
    desc = arcpy.Describe(path)
    count = int(arcpy.management.GetCount(path)[0])

    msg(f"\n{label}")
    msg(f"  Path: {path}")
    msg(f"  Shape type: {desc.shapeType}")
    msg(f"  Feature count: {count}")
    msg(f"  Spatial reference: {desc.spatialReference.name}")

    return desc, count


def require_shape_type(path, label, allowed_shape_types):
    desc = arcpy.Describe(path)

    if isinstance(allowed_shape_types, str):
        allowed_shape_types = [allowed_shape_types]

    if desc.shapeType not in allowed_shape_types:
        raise ValueError(
            f"{label} has invalid geometry type.\n"
            f"  Expected: {allowed_shape_types}\n"
            f"  Found: {desc.shapeType}\n"
            f"  Path: {path}"
        )

    return desc


# ============================================================
# MAIN SCRIPT
# ============================================================

try:
    t0 = time.time()

    msg("Starting initial zones setup workflow...")

    # --------------------------------------------------------
    # Check geodatabases / inputs
    # --------------------------------------------------------

    check_exists(roads_gdb, "roads_gdb")
    check_exists(zones_gdb, "zones_gdb")

    check_exists(roads_9_fc, "roads_9")
    check_exists(extent_hull_fc, "extent_hull")
    check_exists(raw_waterways_fc, "raw OSM waterways layer")
    check_exists(raw_geoboundaries_fc, "raw geoBoundaries ADM2 layer")

    roads_desc, roads_count = describe_basic(roads_9_fc, "Input roads_9")
    extent_hull_desc, extent_hull_count = describe_basic(extent_hull_fc, "Input extent_hull")
    raw_waterways_desc, raw_waterways_count = describe_basic(raw_waterways_fc, "Input raw OSM waterways")
    raw_geoboundaries_desc, raw_geoboundaries_count = describe_basic(raw_geoboundaries_fc, "Input raw geoBoundaries ADM2")

    require_shape_type(roads_9_fc, "roads_9", "Polyline")
    require_shape_type(extent_hull_fc, "extent_hull", "Polygon")
    require_shape_type(raw_waterways_fc, "raw OSM waterways", "Polyline")
    require_shape_type(raw_geoboundaries_fc, "raw geoBoundaries ADM2", "Polygon")

    # --------------------------------------------------------
    # Warn if CRS differs
    # --------------------------------------------------------

    input_srs = {
        "roads_9": roads_desc.spatialReference.name,
        "extent_hull": extent_hull_desc.spatialReference.name,
        "raw_waterways": raw_waterways_desc.spatialReference.name,
        "raw_geoboundaries": raw_geoboundaries_desc.spatialReference.name,
    }

    if len(set(input_srs.values())) > 1:
        msg("\nWARNING: Not all inputs have the same spatial reference.")
        msg("ArcGIS may project on the fly in some tools, but geoprocessing is safer when inputs are already aligned.")
        for name, sr_name in input_srs.items():
            msg(f"  {name}: {sr_name}")

    # --------------------------------------------------------
    # Clean existing outputs / temporary files
    # --------------------------------------------------------

    msg("\nCleaning previous outputs/intermediates if they exist...")

    delete_if_exists(waterways_clip_fc)
    delete_if_exists(geoboundaries_selection_fc)
    delete_if_exists(geoboundaries_line_fc)
    delete_if_exists(extent_hull_line_fc)
    delete_if_exists(zones_1_fc)
    delete_if_exists(merged_lines_fc)
    delete_if_exists(clipped_merged_lines_fc)

    # --------------------------------------------------------
    # Step 1: Clip raw waterways by extent_hull
    # --------------------------------------------------------

    msg("\nStep 1: Clipping raw OSM waterways by extent_hull...")

    arcpy.analysis.Clip(
        in_features=raw_waterways_fc,
        clip_features=extent_hull_fc,
        out_feature_class=waterways_clip_fc
    )

    waterways_clip_desc, waterways_clip_count = describe_basic(
        waterways_clip_fc,
        "Created waterways_clip"
    )

    require_shape_type(waterways_clip_fc, "waterways_clip", "Polyline")

    # --------------------------------------------------------
    # Step 2: Select geoBoundaries polygons that intersect extent_hull
    # --------------------------------------------------------

    msg("\nStep 2: Selecting geoBoundaries polygons that intersect extent_hull...")

    if arcpy.Exists(geoboundaries_lyr):
        arcpy.management.Delete(geoboundaries_lyr)

    arcpy.management.MakeFeatureLayer(
        in_features=raw_geoboundaries_fc,
        out_layer=geoboundaries_lyr
    )

    arcpy.management.SelectLayerByLocation(
        in_layer=geoboundaries_lyr,
        overlap_type="INTERSECT",
        select_features=extent_hull_fc,
        search_distance=None,
        selection_type="NEW_SELECTION"
    )

    selected_count = int(arcpy.management.GetCount(geoboundaries_lyr)[0])
    msg(f"  Selected geoBoundaries features: {selected_count}")

    if selected_count == 0:
        raise RuntimeError(
            "No geoBoundaries features intersected extent_hull. "
            "Check CRS alignment and input paths."
        )

    arcpy.management.CopyFeatures(
        in_features=geoboundaries_lyr,
        out_feature_class=geoboundaries_selection_fc
    )

    geoboundaries_selection_desc, geoboundaries_selection_count = describe_basic(
        geoboundaries_selection_fc,
        "Created geoboundaries_selection"
    )

    require_shape_type(geoboundaries_selection_fc, "geoboundaries_selection", "Polygon")

    # --------------------------------------------------------
    # Step 3: Convert geoboundaries_selection polygons to lines
    # --------------------------------------------------------

    msg("\nStep 3: Converting geoboundaries_selection polygons to geoboundaries_line...")

    arcpy.management.PolygonToLine(
        in_features=geoboundaries_selection_fc,
        out_feature_class=geoboundaries_line_fc,
        neighbor_option="IGNORE_NEIGHBORS"
    )

    geoboundaries_line_desc, geoboundaries_line_count = describe_basic(
        geoboundaries_line_fc,
        "Created geoboundaries_line"
    )

    require_shape_type(geoboundaries_line_fc, "geoboundaries_line", "Polyline")

    # --------------------------------------------------------
    # Step 4: Convert extent_hull polygon to line
    # --------------------------------------------------------

    msg("\nStep 4: Converting extent_hull polygon to extent_hull_line...")

    arcpy.management.PolygonToLine(
        in_features=extent_hull_fc,
        out_feature_class=extent_hull_line_fc,
        neighbor_option="IGNORE_NEIGHBORS"
    )

    extent_hull_line_desc, extent_hull_line_count = describe_basic(
        extent_hull_line_fc,
        "Created extent_hull_line"
    )

    require_shape_type(extent_hull_line_fc, "extent_hull_line", "Polyline")

    # --------------------------------------------------------
    # Step 5: Merge roads_9, waterways_clip, extent_hull_line,
    #         and geoboundaries_line
    # --------------------------------------------------------

    msg("\nStep 5: Merging roads_9, waterways_clip, extent_hull_line, and geoboundaries_line...")

    line_inputs = [
        roads_9_fc,
        waterways_clip_fc,
        extent_hull_line_fc,
        geoboundaries_line_fc,
    ]

    arcpy.management.Merge(
        inputs=line_inputs,
        output=merged_lines_fc
    )

    merged_desc, merged_count = describe_basic(
        merged_lines_fc,
        "Created merged lines"
    )

    require_shape_type(merged_lines_fc, "merged lines", "Polyline")

    # --------------------------------------------------------
    # Step 6: Clip merged lines by extent_hull
    # --------------------------------------------------------

    msg("\nStep 6: Clipping merged lines by extent_hull...")

    arcpy.analysis.Clip(
        in_features=merged_lines_fc,
        clip_features=extent_hull_fc,
        out_feature_class=clipped_merged_lines_fc
    )

    clipped_desc, clipped_count = describe_basic(
        clipped_merged_lines_fc,
        "Created clipped merged lines"
    )

    require_shape_type(clipped_merged_lines_fc, "clipped merged lines", "Polyline")

    # --------------------------------------------------------
    # Step 7: Polygonize clipped merged lines to create zones_1
    # --------------------------------------------------------

    msg("\nStep 7: Polygonizing clipped merged lines to create zones_1...")

    arcpy.management.FeatureToPolygon(
        in_features=[clipped_merged_lines_fc],
        out_feature_class=zones_1_fc,
        cluster_tolerance=None,
        attributes="NO_ATTRIBUTES"
    )

    zones_1_desc, zones_1_count = describe_basic(
        zones_1_fc,
        "Created zones_1"
    )

    require_shape_type(zones_1_fc, "zones_1", "Polygon")

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    msg("\nCleaning temporary scratch outputs...")

    delete_if_exists(merged_lines_fc)
    delete_if_exists(clipped_merged_lines_fc)

    if arcpy.Exists(geoboundaries_lyr):
        arcpy.management.Delete(geoboundaries_lyr)

    elapsed = round((time.time() - t0) / 60, 2)

    msg("\nWorkflow complete.")
    msg(f"  waterways_clip:            {waterways_clip_fc}")
    msg(f"  geoboundaries_selection:   {geoboundaries_selection_fc}")
    msg(f"  geoboundaries_line:        {geoboundaries_line_fc}")
    msg(f"  extent_hull_line:          {extent_hull_line_fc}")
    msg(f"  zones_1:                   {zones_1_fc}")
    msg(f"  zones_1 count:             {zones_1_count}")
    msg(f"  Elapsed time:              {elapsed} minutes")

except Exception as e:
    msg("\nSCRIPT FAILED.")
    msg(str(e))
    msg(traceback.format_exc())
    raise